In [2]:
# import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings 
warnings.filterwarnings('ignore')

In [3]:
# import dataset
df=pd.read_csv('final_data.csv')

In [4]:
# Analysis
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 661 entries, 0 to 660
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        661 non-null    object
 1   company     661 non-null    object
 2   year        661 non-null    int64 
 3   Price       661 non-null    int64 
 4   kms_driven  661 non-null    int64 
 5   fuel_type   661 non-null    object
dtypes: int64(3), object(3)
memory usage: 31.1+ KB


In [5]:
# Divide dataset into x and y
x = df[['company','name','year','kms_driven','fuel_type']]
y = df[['Price']]

In [6]:
# creating pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

In [7]:
ohe = OneHotEncoder()
ohe.fit(x[['company','name','fuel_type']])
ct = make_column_transformer((OneHotEncoder(categories = ohe.categories_),['company','name','fuel_type']),remainder = 'passthrough')
model = LinearRegression()
pipe = make_pipeline(ct,model)

In [8]:
# Divide into train and test, and calculate accuracy
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
scores = []
for i in range(0,101):
    x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.1, random_state = i)
    pipe.fit(x_train, y_train)
    y_pred = pipe.predict(x_test)
    y_pred = pd.DataFrame(data = y_pred, columns = ['prediction'])
    result = pd.concat([y_test.reset_index(drop = True),y_pred], axis = 1)
    score = r2_score(result['Price'], result['prediction'])
    scores.append(score)

In [9]:
# get index of max value
index = np.argmax(scores)

In [10]:
index

np.int64(15)

In [11]:
scores

[0.7747224213680399,
 0.6063457851304739,
 0.6531655915928682,
 0.6678061082435625,
 0.442738556333768,
 0.5449723409815499,
 0.8107680026556207,
 0.6767323806209637,
 0.6315470907567353,
 0.6732748522056811,
 0.5370851427654038,
 0.5016416633734613,
 0.7810722916226149,
 0.6824148302615184,
 0.7464947054423584,
 0.841599012714672,
 0.7733832308203348,
 0.7487337982706477,
 0.5034303939650059,
 0.6878826830187272,
 0.7536970413917652,
 0.40630340363538686,
 0.7492856420985843,
 0.7872602328274365,
 0.726361326737385,
 0.728643717777349,
 0.6648239199326926,
 0.7981306587989867,
 0.6562892410344845,
 0.504449860500193,
 0.6375277137263925,
 0.6416063640630263,
 0.4130520643696802,
 0.6548675665730486,
 0.7131222149405771,
 0.7111245770339183,
 0.7374983918819299,
 0.648168035370198,
 0.6375979637056792,
 0.6180494915620339,
 0.2402531224161194,
 0.6997825099852468,
 0.6897833513613445,
 0.7284162354086434,
 0.6852014286603852,
 0.7206572922196524,
 0.560951120154906,
 0.5426952205050896

In [12]:
# split using best index and train
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.1,random_state = index)
pipe.fit(x_train,y_train)

,steps,"[('columntransformer', ...), ('linearregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehotencoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [13]:
# check for user input
myinput = [['Ford','Figo',2020,80000,'Petrol']]
columns = ['company','name','year','kms_driven','fuel_type']
myinput = pd.DataFrame(data = myinput, columns = columns)
result = pipe.predict(myinput)
print("Predicted Price is :",round(result[0,0]))

Predicted Price is : 455403


In [15]:
import pickle
pickle.dump(pipe, open('pipe.pkl','wb'))